# Data to Harvest: AI for Agricultural Resilience Hackathon: Starter Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/DSU-Hackathon/blob/main/notebooks/starter_notebook.ipynb)

**DSU Hackathon, October 17–18, 2026**

This notebook loads the master dataset, walks through one research question, and leaves space for your own analysis.

One row = one county, one crop, one year. 4 states, 4 crops, 26 years, 21 fields.

Read [`data/data_dictionary.md`](https://github.com/ms-cc-org/DSU-Hackathon/blob/main/data/data_dictionary.md) before you start — it explains every field, every NaN, and every known limitation.

In [1]:
# Google Colab only: fetch the repo's code and data. Does nothing when run locally.
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/ms-cc-org/DSU-Hackathon.git /content/DSU-Hackathon 2>/dev/null || echo "Repo already cloned"
    %cd /content/DSU-Hackathon/notebooks

In [2]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px
from src.data_loader import load_master

df = load_master()
print(f"{len(df):,} rows × {len(df.columns)} columns")
df.head(15)

17,056 rows × 21 columns


,fips,county_name,state_alpha,year,crop,yield_per_acre,yield_status,acres_planted,acres_harvested,yield_anomaly_pct,...,precip_anomaly_pct,tavg_c,extreme_heat_days,gdd,max_drought_severity,weeks_in_d2_plus,mean_dsci,aws_100cm_mm,droughty_pct,nccpi_crop
0,06001,Alameda County,CA,2000,CORN,116.7,reported,600.0,600.0,NaN,...,76.00,20.06,5.0,1507.0,0.0,0.0,0.0,116.899535,62.153917,0.0675
1,06001,Alameda County,CA,2001,CORN,155.0,reported,400.0,400.0,NaN,...,-65.89,20.70,5.0,1591.0,2.0,0.0,69.4,116.899535,62.153917,0.0675
2,06001,Alameda County,CA,2002,CORN,NaN,not_reported,NaN,NaN,NaN,...,-5.11,19.93,6.0,1484.0,1.0,0.0,85.7,116.899535,62.153917,0.0675
3,06001,Alameda County,CA,2003,CORN,152.5,reported,500.0,400.0,NaN,...,12.34,20.58,8.0,1572.0,0.0,0.0,0.0,116.899535,62.153917,0.0675
4,06001,Alameda County,CA,2004,CORN,NaN,not_reported,NaN,NaN,NaN,...,-32.16,20.23,3.0,1537.0,1.0,0.0,59.1,116.899535,62.153917,0.0675
5,06001,Alameda County,CA,2005,CORN,NaN,not_reported,NaN,NaN,NaN,...,154.16,19.78,1.0,1470.0,0.0,0.0,0.0,116.899535,62.153917,0.0675
6,06001,Alameda County,CA,2006,CORN,NaN,not_reported,NaN,NaN,NaN,...,-5.15,20.48,8.0,1550.0,0.0,0.0,0.0,116.899535,62.153917,0.0675
7,06001,Alameda County,CA,2007,CORN,NaN,not_reported,NaN,NaN,NaN,...,-51.53,19.75,2.0,1473.0,3.0,19.0,253.7,116.899535,62.153917,0.0675
8,06001,Alameda County,CA,2008,CORN,NaN,not_reported,NaN,NaN,NaN,...,-99.14,20.06,11.0,1491.0,3.0,10.0,227.2,116.899535,62.153917,0.0675
9,06001,Alameda County,CA,2009,CORN,NaN,not_reported,NaN,NaN,NaN,...,31.82,20.30,6.0,1525.0,3.0,22.0,237.7,116.899535,62.153917,0.0675


In [3]:
# What's in the dataset?
print("States:", sorted(df["state_alpha"].unique()))
print("Crops: ", sorted(df["crop"].unique()))
print("Years: ", df["year"].min(), "–", df["year"].max())
print()

# Rows per state and crop
pd.crosstab(df["state_alpha"], df["crop"], margins=True)

States: ['CA', 'DE', 'IA', 'NE']
Crops:  ['CORN', 'SORGHUM', 'SOYBEANS', 'WHEAT']
Years:  2000 – 2025



crop,CORN,SORGHUM,SOYBEANS,WHEAT,All
state_alpha,,,,,
CA,806,468,0,884,2158
DE,78,78,78,78,312
IA,2574,0,2574,728,5876
NE,2366,2028,2106,2210,8710
All,5824,2574,4758,3900,17056


---
## Worked example: the 2012 Iowa drought

In the summer of 2012, the worst drought in decades hit the Corn Belt. This walkthrough is about the mechanics such as filter, aggregate, plot - with 2012 Iowa corn as the subject. What the numbers say, you can read off the charts

In [4]:
# Filter to Iowa corn with reported yields
ia_corn = df[
    (df["state_alpha"] == "IA")
    & (df["crop"] == "CORN")
    & (df["yield_status"] == "reported")
]

# State average by year
yearly = ia_corn.groupby("year", as_index=False).agg(
    mean_yield=("yield_per_acre", "mean"),
    mean_dsci=("mean_dsci", "mean"),
    mean_anomaly=("yield_anomaly_pct", "mean"),
)

fig = px.bar(
    yearly, x="year", y="mean_yield",
    title="Iowa Corn: Mean County Yield by Year",
    labels={"mean_yield": "Yield (bu/acre)", "year": "Year"},
)
fig.add_vrect(x0=2011.5, x1=2012.5, fillcolor="red", opacity=0.15,
              annotation_text="2012", annotation_position="top left")
fig.show()

In [5]:
# How bad was the drought? Compare DSCI (Drought Severity and Coverage Index)
# 0 = no drought, 500 = entire county in D4 (exceptional) all season

fig = px.bar(
    yearly, x="year", y="mean_dsci",
    title="Iowa Corn: Mean Growing-Season Drought Intensity",
    labels={"mean_dsci": "Mean DSCI", "year": "Year"},
)
fig.add_vrect(x0=2011.5, x1=2012.5, fillcolor="red", opacity=0.15,
              annotation_text="2012", annotation_position="top left")
fig.show()

In [6]:
# County-level scatter: does higher drought intensity predict worse yield anomaly?
# yield_anomaly_pct removes the long-term trend, so we're comparing to each county's own baseline

reported = ia_corn[ia_corn["yield_anomaly_pct"].notna()].copy()

fig = px.scatter(
    reported, x="mean_dsci", y="yield_anomaly_pct",
    color=reported["year"].eq(2012).map({True: "2012", False: "Other years"}),
    opacity=0.5,
    title="Iowa Corn: Yield Anomaly vs Drought Intensity (every county-year)",
    labels={
        "mean_dsci": "Mean DSCI (higher = worse drought)",
        "yield_anomaly_pct": "Yield anomaly (%)",
        "color": "",
    },
)
fig.show()

In [7]:
# Where was 2012 worst? Map each county's yield anomaly.
# counties.geojson uses the 5-digit fips as each feature's id, so locations="fips" lines up.
from src.data_loader import load_counties

counties = load_counties()
ia_2012 = reported[reported["year"] == 2012]

fig = px.choropleth(
    ia_2012, geojson=counties, locations="fips", color="yield_anomaly_pct",
    color_continuous_scale="RdYlGn", range_color=(-40, 40),
    hover_name="county_name",
    title="Iowa Corn: 2012 Yield Anomaly by County",
    labels={"yield_anomaly_pct": "Yield anomaly (%)"},
)
fig.update_geos(fitbounds="locations", visible=False)   # zoom to the counties shown
fig.show()

**What this walkthrough covered:**

- Filtering to one state, one crop, and `yield_status == "reported"`
- Aggregating county rows into a yearly state average with `groupby().agg()`
- Reading one year against its neighbors, then dropping back to county level to test whether a pattern holds beyond that single year
- Mapping county values with `load_counties()` and `px.choropleth`

One field worth understanding before you go further: `yield_anomaly_pct` measures each county against its own long-term trend, so it separates *a bad year* from *a low-yielding county*. See the data dictionary for how it's computed.

---

## Your turn

Three places to start. Each names a question and the fields that bear on it — the approach is yours. Pick one, combine them, or go somewhere else entirely.

### 1. Does soil buffer drought?

Soils differ in how much water they hold for a crop to draw on. Relevant fields: `aws_100cm_mm`, `droughty_pct`, `mean_dsci`, `weeks_in_d2_plus`, `yield_anomaly_pct`. How you define a drought year and what would count as evidence of buffering — is your call. Iowa is rainfed; much of Nebraska is irrigated, and the lowest-storage Nebraska counties are the most irrigated, which the soil fields can't show. The irrigation extension (`load_irrigation()`) has Nebraska's irrigated share, mostly 2000–2018.

In [8]:
# Your analysis here


### 2. Corn vs sorghum in Nebraska

Sorghum is often described as drought-tolerant. Nebraska is the only state here with substantial reported yields for both crops (California and Delaware have sorghum acreage but comparatively few reported yields). Relevant fields: `crop`, `yield_anomaly_pct`, `mean_dsci`, `weeks_in_d2_plus`, `yield_status`. Check your row counts before trusting a comparison. Most Nebraska corn is irrigated and most sorghum is not, so a drought comparison between them is also an irrigated-vs-dryland comparison. The irrigation extension (`load_irrigation()`) has irrigated and dryland yields for Nebraska sorghum through 2007.

In [9]:
# Your analysis here


### 3. Rainfed vs irrigated

Iowa corn is rainfed; California agriculture is heavily irrigated. Relevant fields: `precip_mm`, `precip_anomaly_pct`, `yield_anomaly_pct`, `tavg_c`, `extreme_heat_days`. The two states only share corn and wheat, and the reported-row counts are lopsided — Iowa has 2,470 reported corn rows to California's 261, and California has 385 reported wheat rows to Iowa's 103. About two-thirds of California's reported corn is from before 2010. The crosstab near the top counts all rows, not reported yields, so filter to `yield_status == "reported"` before you pick your comparison.


In [10]:
# Your analysis here


---

### Finer resolution data

The master dataset aggregates weather and drought into one row per county-crop-year. If you want daily or weekly resolution — to build your own season windows, find sub-seasonal patterns, or look at daily extremes — load the source panels:

```python
from src.data_loader import load_daily_weather, load_weekly_drought, load_soil, load_nass_raw, load_modis, load_irrigation

weather = load_daily_weather()    # 2.5M rows: daily tmax/tmin/precip per county
drought = load_weekly_drought()   # 343K rows: weekly D0–D4 per county
soil    = load_soil()             # 253 rows: one per county (static)
nass    = load_nass_raw()         # 35K rows: NASS yield/acres in long format
modis   = load_modis()            # 5,980 rows: annual NDVI/EVI per county (extension)
irrigation = load_irrigation()    # 3,716 rows: irrigated share and yields, mostly Nebraska (extension)
```

All join on `fips` (plus `date`, `map_date`, `year` or `crop` + `year`). See the data dictionary for details.